In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_esquema = "movie_gold"
v_tabla = "results_country_prod_company"
v_partition = "file_date"
v_merge_condition = "target.movie_id = source.movie_id and target.country_id = source.country_id and target.company_id = source.company_id"

In [0]:
#DataFrames con la data con la cual se va a trabajar, incluye filtros y campos

##Tabla de carga completa
country_df = spark.read.table("movie_silver.countries")

#Tablas con data particionada
movies_df = spark.read.table("movie_silver.movies")\
                      .filter(
                               (col("file_date") >= f"{v_file_date}")
                             )\
                      .select("movie_id", "title", "budget", "revenue", "duration_time", "release_date", "year_release_date")
##
production_country_df = spark.read.table("movie_silver.productions_countries")\
                                  .filter(
                                          (col("file_date") == f"{v_file_date}")
                                         )
##
movie_company_df = spark.read.table("movie_silver.movies_companies")\
                              .filter(
                                      (col("file_date") == f"{v_file_date}")
                                     )
##
production_company_df = spark.read.table("movie_silver.productions_companies")\
                                  .filter(
                                           (col("file_date") == f"{v_file_date}")
                                         )



In [0]:
#DF country
country_prod_coun_df = country_df.join(production_country_df,
                                   country_df.country_id == production_country_df.country_id
                                   , "inner")\
                              .select(production_country_df.movie_id, country_df.country_id, country_df.country_name)

#DF company
prod_comp_mov_comp_df = production_company_df.join(movie_company_df,
                                              movie_company_df.company_id == production_company_df.company_id
                                              , "inner")\
                                    .select(movie_company_df.movie_id, production_company_df.company_id, production_company_df.company_name)


#Filtra df_movies
movie_filter_df = movies_df.filter(
                                    (col("year_release_date") >= "2010")
                                   )

#df movies y country
result_movies_country_prod_company_df = movie_filter_df.join(country_prod_coun_df,
                                                            movie_filter_df.movie_id == country_prod_coun_df.movie_id
                                                            , "inner")\
                                                       .join (prod_comp_mov_comp_df, 
                                                              movie_filter_df.movie_id == prod_comp_mov_comp_df.movie_id
                                                               , "inner")
                                                            


In [0]:
#Seleccionamos las columnas y ordenamos
results_df = result_movies_country_prod_company_df.select(movie_filter_df.movie_id, "country_id", "company_id" ,"title", "budget", "revenue",
                                                            "duration_time", "release_date", "country_name", "company_name") 

results_df = add_ingestion_date(results_df)
results_df = add_env(results_df)
final_df = add_file_date (results_df)



In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Guardamos en la capa gold 

#results_country_prod_company_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {results_country_prod_company_df.count()} registros en la tabla {v_esquema}.{v_tabla}")
